In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Auxiliar de aprendizado de ruído

<details>
<summary><b>Versões dos pacotes</b></summary>

O código desta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou versões mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
As técnicas de mitigação de erros [PEA](./error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) e [PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) utilizam um componente de aprendizado de ruído baseado em um [modelo de ruído de Pauli-Lindblad](https://arxiv.org/abs/2201.09866), que normalmente é gerenciado durante a execução após o envio de um ou mais jobs pelo `qiskit-ibm-runtime`, sem acesso local ao modelo de ruído ajustado. No entanto, a partir da versão 0.27.1 do `qiskit-ibm-runtime`, foram criadas as classes [`NoiseLearner`](../api/qiskit-ibm-runtime/noise-learner) e [`NoiseLearnerOptions`](../api/qiskit-ibm-runtime/options-noise-learner-options) para obter os resultados desses experimentos de aprendizado de ruído. Esses resultados podem ser armazenados localmente como um `NoiseLearnerResult` e usados como entrada em experimentos posteriores. Esta página fornece uma visão geral do uso e das opções disponíveis.

## Visão geral
A classe `NoiseLearner` realiza experimentos que caracterizam processos de ruído com base em um modelo de ruído de Pauli-Lindblad para um (ou mais) circuits. Ela possui um método `run()` que executa os experimentos de aprendizado e recebe como entrada uma lista de circuits ou um [PUB](./primitive-input-output#overview-of-pubs), retornando um `NoiseLearnerResult` contendo os canais de ruído aprendidos e metadados sobre os jobs enviados. Abaixo está um trecho de código demonstrando o uso do programa auxiliar.

In [1]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2
from qiskit_ibm_runtime.noise_learner import NoiseLearner
from qiskit_ibm_runtime.options import (
    NoiseLearnerOptions,
    ResilienceOptionsV2,
    EstimatorOptions,
)

# Build a circuit with two entangling layers
num_qubits = 27
edges = list(CouplingMap.from_line(num_qubits, bidirectional=False))
even_edges = edges[::2]
odd_edges = edges[1::2]

circuit = QuantumCircuit(num_qubits)
for pair in even_edges:
    circuit.cx(pair[0], pair[1])
for pair in odd_edges:
    circuit.cx(pair[0], pair[1])

# Choose a backend to run on
service = QiskitRuntimeService()
backend = service.least_busy()

# Transpile the circuit for execution
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
circuit_to_learn = pm.run(circuit)

# Instantiate a NoiseLearner object and execute the noise learning program
learner = NoiseLearner(mode=backend)
job = learner.run([circuit_to_learn])
noise_model = job.result()

O `NoiseLearnerResult.data` resultante é uma lista de objetos [`LayerError`](../api/qiskit-ibm-runtime/utils-noise-learner-result-layer-error) contendo o [modelo de ruído](https://arxiv.org/abs/2201.09866) para cada camada de entrelaçamento individual que pertence ao(s) circuit(s) alvo. Cada `LayerError` armazena as informações da camada, na forma de um circuit e um conjunto de rótulos de Qubits, juntamente com o [`PauliLindbladError`](../api/qiskit-ibm-runtime/utils-noise-learner-result-pauli-lindblad-error) para o modelo de ruído aprendido para a camada em questão.

In [2]:
import numpy

print(
    f"Noise learner result contains {len(noise_model.data)} entries"
    f" and has the following type:\n {type(noise_model)}\n"
)
print(
    f"Each element of `NoiseLearnerResult` then contains"
    f" an object of type:\n {type(noise_model.data[0])}\n"
)
# Results are truncated
with numpy.printoptions(threshold=200):
    print(
        f"And each of these `LayerError` objects possess"
        f" data on the generators for the error channel: \n{noise_model.data[0].error.generators}\n"
    )
# Results are truncated
with numpy.printoptions(threshold=200):
    print(
        f"Along with the error rates: \n{noise_model.data[0].error.rates}\n"
    )

Noise learner result contains 2 entries and has the following type:
 <class 'qiskit_ibm_runtime.utils.noise_learner_result.NoiseLearnerResult'>

Each element of `NoiseLearnerResult` then contains an object of type:
 <class 'qiskit_ibm_runtime.utils.noise_learner_result.LayerError'>

And each of these `LayerError` objects possess data on the generators for the error channel: 
['IIIIIIIIIIIIIIIIIIIIIIIIIIX', 'IIIIIIIIIIIIIIIIIIIIIIIIIIY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIXI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIXX', 'IIIIIIIIIIIIIIIIIIIIIIIIIXY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIXZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIYI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIYX', 'IIIIIIIIIIIIIIIIIIIIIIIIIYY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIYZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIZI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIZX', 'IIIIIIIIIIIIIIIIIIIIIIIIIZY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIZZ', 'IIIIIIIIIIIIIIIIIIIIIIIIXII',
 'IIIIIIIIIIIIIIIIIIIIIIIIXIX', 'IIIIIIIIIIIIIIIIIIIIIIIIXIY',
 'IIIIIIIIIIIIIIIIIIIIIIIIXIZ', 'IIIIIIIIIIIIIIIIIIIIII

The `LayerError.error` attribute of the noise learning result contains the generators and error rates of the fitted Pauli Lindblad model, which has the form

$$ \Lambda(\rho) = \exp{\sum_j r_j \left(P_j \rho P_j^\dagger - \rho\right)}, $$

where the $r_j$ are the `LayerError.rates` and $P_j$ are the Pauli operators specified in `LayerError.generators`.

### Noise learning options

You can choose among several options to input when you instantiate a `NoiseLearner` object. These options are encapsulated by the `qiskit_ibm_runtime.options.NoiseLearnerOptions` class and include the ability to specify the maximum layers to learn, number of randomizations, and the twirling strategy, among others. Refer to the [`NoiseLearnerOptions`](/docs/api/qiskit-ibm-runtime/options-noise-learner-options) API documentation for detailed information.

Following is a simple example that shows how to use the `NoiseLearnerOptions` in a `NoiseLearner` experiment:

In [3]:
# Build a GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))
# Choose a backend to run on
service = QiskitRuntimeService()
backend = service.least_busy()

# Transpile the circuit for execution
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
circuit_to_run = pm.run(circuit_to_learn)

# Instantiate a NoiseLearnerOptions object
learner_options = NoiseLearnerOptions(
    max_layers_to_learn=3, num_randomizations=32, twirling_strategy="all"
)

# Instantiate a NoiseLearner object and execute the noise learning program
learner = NoiseLearner(mode=backend, options=learner_options)
job = learner.run([circuit_to_run])
noise_model = job.result()

O atributo `LayerError.error` do resultado do aprendizado de ruído contém os geradores e as taxas de erro do modelo de Pauli Lindblad ajustado, que tem a forma

$$ \Lambda(\rho) = \exp{\sum_j r_j \left(P_j \rho P_j^\dagger - \rho\right)}, $$

onde os $r_j$ são os `LayerError.rates` e $P_j$ são os operadores de Pauli especificados em `LayerError.generators`.
## Opções de aprendizado de ruído
Você pode escolher entre várias opções ao instanciar um objeto `NoiseLearner`. Essas opções são encapsuladas pela classe `qiskit_ibm_runtime.options.NoiseLearnerOptions` e incluem a possibilidade de especificar o número máximo de camadas a aprender, o número de randomizações e a estratégia de twirling, entre outras. Consulte a documentação da API sobre [`NoiseLearnerOptions`](../api/qiskit-ibm-runtime/options-noise-learner-options) para informações mais detalhadas.

Abaixo está um exemplo simples mostrando como usar o `NoiseLearnerOptions` em um experimento com `NoiseLearner`:

In [4]:
# Pass the noise model to the `estimator.options` attribute directly
estimator = EstimatorV2(mode=backend)
estimator.options.resilience.layer_noise_model = noise_model

In [5]:
# Specify options through a ResilienceOptionsV2 object
resilience_options = ResilienceOptionsV2(layer_noise_model=noise_model)
estimator_options = EstimatorOptions(resilience=resilience_options)
estimator = EstimatorV2(mode=backend, options=estimator_options)

In [6]:
# Specify options by using a dictionary
options_dict = {
    "resilience_level": 2,
    "resilience": {"layer_noise_model": noise_model},
}

estimator = EstimatorV2(mode=backend, options=options_dict)

After the noise model is passed into the `EstimatorV2` object, it can be used to run workloads and perform error mitigation as normal.

## NoiseLearnerV3

### Overview

Similar to `NoiseLearner`, the `NoiseLearnerV3` class performs experiments that characterize noise processes based on a Pauli-Lindblad noise model for one or more circuits. Its `run()` method takes a list of instructions, each of which must be a twirled-annotated [`BoxOp`](/docs/api/qiskit/qiskit.circuit.BoxOp) containing [ISA](/docs/guides/transpile#instruction-set-architecture) operations.


The result of a `NoiseLearnerV3` job contains a list of [`NoiseLearnerV3Result`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/qiskit_ibm_runtime.noise_learner_v3.NoiseLearnerV3Result.html) objects, one for each input instruction.
The following code shows how to use the helper program.

In [7]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic.utils import find_unique_box_instructions


# Build a circuit with two entangling layers
num_qubits = 27
edges = list(CouplingMap.from_line(num_qubits, bidirectional=False))
even_edges = edges[::2]
odd_edges = edges[1::2]

circuit = QuantumCircuit(num_qubits)
for pair in even_edges:
    circuit.cx(pair[0], pair[1])
for pair in odd_edges:
    circuit.cx(pair[0], pair[1])

# Choose a backend to run on
service = QiskitRuntimeService()
backend = service.least_busy()

# Transpile the circuit for execution
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_circuit = pm.run(circuit)

# Run the boxing pass manager to group instructions into annotated boxes
boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=False,
    inject_noise_targets="gates",  # no measurement mitigation
    inject_noise_strategy="uniform_modification",
)
boxed_circuit = boxing_pm.run(isa_circuit)

# Find unique boxed instructions
unique_box_instructions = find_unique_box_instructions(boxed_circuit.data)
print(f"Found {len(unique_box_instructions)} unique layers")
print(
    f"Each instruction is of type {type(unique_box_instructions[0].operation)}"
)
print(
    f"And has annotations: {unique_box_instructions[0].operation.annotations}"
)

# Instantiate a NoiseLearnerV3 object and execute the noise learning program
learner = NoiseLearnerV3(backend)
learner.options.shots_per_randomization = 128
learner.options.num_randomizations = 32
learner_job = learner.run(unique_box_instructions)
learner_result = learner_job.result()

Found 3 unique layers
Each instruction is of type <class 'qiskit.circuit.controlflow.box.BoxOp'>
And has annotations: [Twirl(group='pauli', dressing='left', decomposition='rzsx'), InjectNoise(ref='r789B', modifier_ref='', site='before')]


Depois que o modelo de ruído for passado para o objeto `EstimatorV2`, ele pode ser usado para executar cargas de trabalho e realizar a mitigação de erros normalmente.
## Próximos passos
> **Tip:** - Leia mais sobre [como configurar a mitigação de erros](configure-error-mitigation).
>     - Consulte a [referência da API EstimatorOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) e a [referência da API ResilienceOptionsV2](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2).
>     - Saiba mais sobre as [técnicas de mitigação e supressão de erros](error-mitigation-and-suppression-techniques) disponíveis no Qiskit Runtime.
>     - Veja como [especificar opções](specify-runtime-options) para as primitivas do Qiskit Runtime.
>     - Leia [Migrar para primitivas V2](/guides/v2-primitives).

In [8]:
print(
    f"The Noise learner V3 result contains {len(learner_result)} entries"
    f" and each has the following type:\n {type(learner_result[0])}\n"
)
noise_map = learner_result[0].to_pauli_lindblad_map()
print(
    f"After converting to PauliLindbladMap, you can extract data "
    f" on the generators for the error channel (truncated to 3): \n{noise_map.generators()[:3]}\n"
)
with numpy.printoptions(threshold=20):
    print(
        f"Along with the error rates (truncated to 3): \n{noise_map.rates[:3]}\n"
    )

The Noise learner V3 result contains 3 entries and each has the following type:
 <class 'qiskit_ibm_runtime.results.noise_learner_v3.NoiseLearnerV3Result'>

After converting to PauliLindbladMap, you can extract data  on the generators for the error channel (truncated to 3): 
<QubitSparsePauliList with 3 elements on 27 qubits: [X_0, Y_0, Z_0]>

Along with the error rates (truncated to 3): 
[0.00026 0.00032 0.00023]



### Noise learning options

`NoiseLearnerV3` supports several options, including the number of randomizations and layer pair depth, among others. Similar to the primitives, you can specify the options during or after instantiating the `NoiseLearnerV3` object. The previous code example  demonstrated how to set the  `shots_per_randomization` and `num_randomizations` options. Refer to the [`NoiseLearnerV3Options`](https://qiskit.github.io/qiskit-ibm-runtime/stubs/qiskit_ibm_runtime.options.NoiseLearnerV3Options.html#qiskit_ibm_runtime.options.NoiseLearnerV3Options) API documentation for detailed information.

### Input a noise model to Executor

Executor follows the design intents specified in circuit annotations (in the form of a samplex) and options. `InjectNoise` is the annotation for specifying where to inject noise, and the `pauli_lindblad_maps` samplex argument specifies which noise map to use.

The circuit in the previous example runs through the boxing pass manager, which groups instructions into annotated boxes. The relevant code is added here for ease of understanding.
- `inject_noise_targets=”gates”` specifies to add the `InjectNoise` annotations to boxes that contain entanglers.
- `inject_noise_strategy="uniform_modification"` specifies to assign the same `ref` and `modifier_ref` to all equivalent boxes with `InjectNoise` annotations.
      - `InjectNoise.ref` is a unique identifier used to assign a noise model to that box.
      - `InjectNoise.modifier_ref` allows scaling the noise model assigned to a box by multiplicative factors.

```python
boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=False,
    inject_noise_targets="gates",  # no measurement mitigation
    inject_noise_strategy="uniform_modification",
)
```

The circuit from the previous example contains three boxes, two of which contain `InjectNoise` annotations with different `ref` attributes (since they are not equivalent).

In [ ]:
# box_circuit comes from the example above
for idx, instruction in enumerate(boxed_circuit):
    # The `InjectNoise` annotation defines which boxes to inject noise.
    print(f"Annotations of box #{idx}: {instruction.operation.annotations}\n")

Annotations of box #0: [Twirl(group='pauli', dressing='left', decomposition='rzsx'), InjectNoise(ref='r789B', modifier_ref='r789B', site='before')]

Annotations of box #1: [Twirl(group='pauli', dressing='left', decomposition='rzsx'), InjectNoise(ref='r054B', modifier_ref='r054B', site='before')]

Annotations of box #2: [Twirl(group='pauli', dressing='right', decomposition='rzsx')]



The result of the `NoiseLearnerV3` job must be converted to a dictionary before being passed to Executor. This dictionary's keys are the `InjectNoise.ref` attributes and the values are the corresponding noise maps. This mapping tells Executor which noise models to inject where.

The following code shows how to take the circuit and the `NoiseLearnerV3` result from the previous example and pass them to Executor, which will generate the circuit variants with the injected noise models and execute them on hardware.

In [10]:
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from samplomatic import build

# Generate a quantum program
program = QuantumProgram(shots=1000)

# Build the template circuit and samplex pair
template_circuit, samplex = build(boxed_circuit)

# Convert the NoiseLearnerV3 result to a dictionary
noise_maps = learner_result.to_dict(
    instructions=unique_box_instructions, require_refs=False
)

# Append the samplex item and execute
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    samplex_arguments={
        "pauli_lindblad_maps": noise_maps,
    },
)

executor = Executor(backend)
executor_job = executor.run(program)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Review the [EstimatorOptions API reference](/docs/api/qiskit-ibm-runtime/options-estimator-options) and [ResilienceOptionsV2 API reference](/docs/api/qiskit-ibm-runtime/options-resilience-options-v2).
    - Learn more about [Error mitigation and suppression techniques](error-mitigation-and-suppression-techniques) that are available through Qiskit Runtime.
    - Learn how to implement [Estimator noise management](/docs/guides/estimator-noise-management).
    - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>